# Makine Öğrenmesi Ara Ödevi

**Konu:** Müşteri Ayrılma Tahmini ile Temel Makine Öğrenmesi Akışı

Bu notebook ödevde istenen maddeleri ayrı hücrelerde gösterir. Hücreler çalıştırılmış ve çıktılar kaydedilmiştir.

## 1. Docstring ve kütüphaneler

In [1]:
"""
Amaç: Sentetik müşteri verileriyle churn tahmini yapmak.
Kütüphaneler: numpy, pandas, matplotlib, scikit-learn
Çalıştırma: Hücreleri yukarıdan aşağıya çalıştırın.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
RANDOM_STATE = 42

## 2. Sentetik veri setini oluşturma

In [2]:
def sigmoid(x): return 1/(1+np.exp(-x))
def veri_seti_olustur(n=500, seed=42):
    r=np.random.default_rng(seed)
    yas=r.integers(18,71,n); gelir=np.clip(r.normal(52000,18000,n),15000,120000).round()
    abonelik=r.integers(1,121,n); destek=np.clip(r.poisson(2,n),0,9); kullanim=np.clip(r.normal(42,18,n),3,120).round(1)
    sehir=r.choice(["Ankara","İstanbul","İzmir","Bursa","Antalya"],n,p=[.22,.32,.18,.15,.13])
    uyelik=r.choice(["Basic","Standard","Premium"],n,p=[.42,.38,.20])
    skor=-2+.7*(destek>=3)+1.1*(destek>=5)+1.3*(abonelik<12)+.7*((abonelik>=12)&(abonelik<24))+.9*(uyelik=="Basic")-.8*(uyelik=="Premium")+.7*(gelir<35000)-.4*(gelir>75000)+.7*(kullanim<20)+.3*(yas<25)+r.normal(0,.25,n)
    churn=r.binomial(1,sigmoid(skor))
    d=pd.DataFrame({"yas":yas,"gelir":gelir,"abonelik_suresi":abonelik,"destek_talebi_sayisi":destek,"aylik_kullanim_saati":kullanim,"sehir":sehir,"uyelik_tipi":uyelik,"churn":churn})
    for c,o in {"gelir":.05,"aylik_kullanim_saati":.03,"sehir":.04}.items(): d.loc[r.choice(d.index,int(n*o),replace=False),c]=np.nan
    return d

## 3. DataFrame oluşturma ve CSV kaydetme

In [3]:
df_olusturulan=veri_seti_olustur()
df_olusturulan.to_csv("musteri_ayrilma_verisi.csv",index=False,encoding="utf-8-sig")
print("CSV dosyası oluşturuldu.")

CSV dosyası oluşturuldu.


## 4. CSV dosyasını pandas ile okuma

In [4]:
df=pd.read_csv("musteri_ayrilma_verisi.csv")
print("Veri seti başarıyla okundu.")

Veri seti başarıyla okundu.


## 5. İlk satırları inceleme

In [5]:
print(df.head())

   yas    gelir  abonelik_suresi  ...     sehir  uyelik_tipi churn
0   22  50303.0               11  ...  İstanbul     Standard     1
1   59  20361.0               69  ...     İzmir     Standard     1
2   52  25593.0               28  ...  İstanbul     Standard     0
3   41  90326.0               71  ...  İstanbul      Premium     0
4   40  28826.0               83  ...   Antalya        Basic     1

[5 rows x 8 columns]


## 6. Satır ve sütun sayısı

In [6]:
print("Veri setinin boyutu:",df.shape)
print("Satır sayısı:",df.shape[0])
print("Sütun sayısı:",df.shape[1])

Veri setinin boyutu: (500, 8)
Satır sayısı: 500
Sütun sayısı: 8


## 7. Hedef değişken dağılımı

In [7]:
print(df["churn"].value_counts().sort_index())
print(df["churn"].value_counts(normalize=True).sort_index().round(3))

churn
0    369
1    131
Name: count, dtype: int64
churn
0    0.738
1    0.262
Name: proportion, dtype: float64


## 8. Eksik değer kontrolü

In [8]:
print(df.isnull().sum())
print("Toplam eksik değer:",df.isnull().sum().sum())

yas                      0
gelir                   25
abonelik_suresi          0
destek_talebi_sayisi     0
aylik_kullanim_saati    15
sehir                   20
uyelik_tipi              0
churn                    0
dtype: int64
Toplam eksik değer: 60


## 9. Öznitelik üretme

In [9]:
df["gelir_grubu"]=pd.cut(df["gelir"],[0,35000,70000,np.inf],labels=["Düşük","Orta","Yüksek"]).astype(object)
df["destek_talebi_var_mi"]=np.where(df["destek_talebi_sayisi"]>0,"Evet","Hayır")
df["abonelik_yili"]=(df["abonelik_suresi"]/12).round(1)
print(df[["gelir","gelir_grubu","destek_talebi_var_mi","abonelik_yili"]].head())

     gelir gelir_grubu destek_talebi_var_mi  abonelik_yili
0  50303.0        Orta                 Evet            0.9
1  20361.0       Düşük                 Evet            5.8
2  25593.0       Düşük                 Evet            2.3
3  90326.0      Yüksek                 Evet            5.9
4  28826.0       Düşük                 Evet            6.9


## 10. X ve y değişkenlerini hazırlama

In [10]:
X=df.drop(columns="churn"); y=df["churn"]
print("X boyutu:",X.shape)
print("y boyutu:",y.shape)

X boyutu: (500, 10)
y boyutu: (500,)


## 11. Train-validation-test bölme

In [11]:
X_train,X_tmp,y_train,y_tmp=train_test_split(X,y,test_size=.30,random_state=42,stratify=y)
X_val,X_test,y_val,y_test=train_test_split(X_tmp,y_tmp,test_size=.50,random_state=42,stratify=y_tmp)
print("Train:",X_train.shape,y_train.shape)
print("Validation:",X_val.shape,y_val.shape)
print("Test:",X_test.shape,y_test.shape)

Train: (350, 10) (350,)
Validation: (75, 10) (75,)
Test: (75, 10) (75,)


## 12. Sayısal ve kategorik sütunları belirleme

In [12]:
sayisal=X.select_dtypes(include=np.number).columns.tolist(); kategorik=X.select_dtypes(exclude=np.number).columns.tolist()
print(sayisal)
print(kategorik)

['yas', 'gelir', 'abonelik_suresi', 'destek_talebi_sayisi', 'aylik_kullanim_saati', 'abonelik_yili']
['sehir', 'uyelik_tipi', 'gelir_grubu', 'destek_talebi_var_mi']


## 13. Eksik değer doldurma, ölçekleme ve One-Hot Encoding

In [13]:
num=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
cat=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))])
on_isleme=ColumnTransformer([("num",num,sayisal),("cat",cat,kategorik)])
print("Ön işleme Pipeline'ı hazırlandı.")

Ön işleme Pipeline'ı hazırlandı.


## 14. Modelleri tanımlama

In [14]:
modeller={"Logistic Regression":LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42),"KNN":KNeighborsClassifier(n_neighbors=9),"Decision Tree":DecisionTreeClassifier(max_depth=4,class_weight="balanced",random_state=42)}
print(list(modeller.keys()))

['Logistic Regression', 'KNN', 'Decision Tree']


## 15. Modelleri eğitme ve validation metrikleri

In [15]:
sonuclar=[]; egitilen={}
for ad,m in modeller.items():
    p=Pipeline([("on_isleme",on_isleme),("model",m)]); p.fit(X_train,y_train); t=p.predict(X_val)
    sonuclar.append({"model":ad,"accuracy":accuracy_score(y_val,t),"precision":precision_score(y_val,t,zero_division=0),"recall":recall_score(y_val,t,zero_division=0),"f1_score":f1_score(y_val,t,zero_division=0)}); egitilen[ad]=p
print("Bütün modeller eğitildi.")

Bütün modeller eğitildi.


## 16. Validation sonuçlarını karşılaştırma

In [16]:
val_df=pd.DataFrame(sonuclar).set_index("model").sort_values("f1_score",ascending=False)
print(val_df.round(3))

                     accuracy  precision  recall  f1_score
model                                                     
Decision Tree           0.720      0.476    0.50     0.488
Logistic Regression     0.640      0.360    0.45     0.400
KNN                     0.707      0.250    0.05     0.083


## 17. En iyi modeli seçme ve yeniden eğitme

In [17]:
en_iyi=val_df.index[0]
X_final=pd.concat([X_train,X_val]); y_final=pd.concat([y_train,y_val])
final_model=clone(egitilen[en_iyi]); final_model.fit(X_final,y_final)
print("Seçilen model:",en_iyi)

Seçilen model: Decision Tree


## 18. Test setinde metrikleri hesaplama

In [18]:
test_tahmin=final_model.predict(X_test)
metrikler={"accuracy":accuracy_score(y_test,test_tahmin),"precision":precision_score(y_test,test_tahmin,zero_division=0),"recall":recall_score(y_test,test_tahmin,zero_division=0),"f1_score":f1_score(y_test,test_tahmin,zero_division=0)}
for k,v in metrikler.items(): print(f"{k:10s}: {v:.3f}")

accuracy  : 0.667
precision : 0.350
recall    : 0.368
f1_score  : 0.359


## 19. Confusion Matrix

In [19]:
cm=confusion_matrix(y_test,test_tahmin)
print("Confusion Matrix:")
print(cm)
ConfusionMatrixDisplay(cm,display_labels=["Kalır (0)","Ayrılır (1)"]).plot(values_format="d")
plt.title(f"Confusion Matrix - {en_iyi}"); plt.show()

Confusion Matrix:
[[43 13]
 [12  7]]


## 20. Kısa sonuç yorumu

In [20]:
print(f"Validation F1-score değerine göre en iyi model {en_iyi} oldu. Test F1-score değeri {metrikler['f1_score']:.3f}. Sentetik veride churn eşik tabanlı ilişkilerle üretildiği için Decision Tree bu ilişkileri daha iyi yakalamış olabilir.")

Validation F1-score değerine göre en iyi model Decision Tree oldu. Test F1-score değeri 0.359. Sentetik veride churn eşik tabanlı ilişkilerle üretildiği için Decision Tree bu ilişkileri daha iyi yakalamış olabilir.
